<a href="https://colab.research.google.com/github/prasannakumar9i/ev-llm-diagnostic-assistant/blob/main/ev_rag_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EV LLM Diagnostic Assistant (RAG Project)

Author: Sunny  
Project: EV AI Diagnostic Assistant  
Tools: Python, LangChain, FAISS, HuggingFace

## Day 1 – Project Setup

Goal:
- Create GitHub repository
- Connect Google Colab
- Initialize project notebook

#Day 2 Install libraries

In [ ]:
!pip install langchain
!pip  install sentence-transformers
!pip install faiss-cpu
!pip install pypdf
!pip install transformers

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import faiss

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Embedding model loaded successfully")

In [ ]:
sentence = "EV battery draining quickly"

embedding = model.encode(sentence)

print(len(embedding))

# Day 3 — Download EV Repair Manuals Dataset

In this step, we collect EV repair manuals in PDF format.
These manuals will act as the knowledge base for the AI diagnostic assistant.
Later we will extract text from these PDFs and convert them into embeddings.

In [ ]:
import os

os.makedirs("ev_manuals", exist_ok=True)

print("EV manuals dataset folder created successfully")

In [ ]:
import os
os.listdir()

# Day 4 — Load PDF Manuals and Extract Text

In this step we upload a PDF manual into Google Colab and use
LangChain's PyPDFLoader to read the document.

Each page of the PDF will be converted into a document object
that will later be split into chunks for the RAG system.

#install pdf Libraries

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!pip install langchain
!pip install langchain-community
!pip install pypdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("harrier-ev-onwers-manual.pdf")

documents = loader.load()

print("Total pages loaded:", len(documents))

In [ ]:
print(documents[0].page_content[:500])

# Day 5 — Split Documents into Text Chunks

In this step we split the extracted manual text into smaller chunks.
Chunking improves retrieval accuracy in RAG systems because the
language model searches smaller text segments instead of full documents.

In [ ]:
!pip install langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
# creating text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=20
)

#split the documents

chunks = text_splitter.split_documents(documents)

print("Total chunks:", len(chunks))


In [ ]:
print(chunks[0].page_content[:1000])


#Day 6 Generate Embeddings

In [ ]:
#packages
!pip install langchain
!pip install sentence-transformers
!pip install langchain-community
!pip install langchain-text-splitters

In [ ]:
# imports
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
#loading the pdf
loader = PyPDFLoader("harrier-ev-onwers-manual.pdf")
documents = loader.load()

In [ ]:
# splitting the text
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

docs = text_splitter.split_documents(documents)

print(len(docs))

In [ ]:
#creating embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2")


In [ ]:
#generating vectors
texts = [doc.page_content for doc in docs]

vectors = embeddings.embed_documents(texts)

print(len(vectors))
print(len(vectors[0]))

#Day 7 creating vector database

In [ ]:
!pip install faiss-cpu

In [ ]:
#importing faiss
from langchain_community.vectorstores import FAISS

In [ ]:
#create vector database
vectorstore = FAISS.from_documents(docs, embeddings)

In [ ]:
#testing
query = "battery not charging"

results = vectorstore.similarity_search(query, k=3)

for r in results:
    print(r.page_content)
    print("------")


In [ ]:
vectorstore.save_local("ev_manual_vector_db")

In [ ]:
query = "battery overheating"

results = vectorstore.similarity_search(query, k=3)

for r in results:
    print(r.page_content)
    print("------")

In [ ]:
query = "charging port problem"

results = vectorstore.similarity_search(query, k=3)

for r in results:
    print(r.page_content)
    print("------")

In [ ]:
from google.colab import files
files.download("ev_manual_vector_db/index.faiss")